In [ ]:
import torchvision.datasets as datasets

# 1. Download/Load the PyTorch MNIST training set
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True)

# 2. Extract and convert images and labels to NumPy
# Images shape will be (60000, 28, 28), Labels shape will be (60000,)
train_images = mnist_trainset.data.numpy()
train_labels = mnist_trainset.targets.numpy()

print(type(train_images), train_images.shape)
X_train = train_images

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

X_train_tensor = torch.from_numpy(X_train).float().div(255).unsqueeze(1)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def create_denoising_ae(dropout_rate):
    encoder = nn.Sequential(
        nn.Dropout(dropout_rate),

        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )

    decoder = nn.Sequential(
        nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),
        nn.ReLU(),

        nn.ConvTranspose2d(16, 1, kernel_size=2, stride=2),
        nn.Sigmoid()
    )

    return nn.Sequential(encoder, decoder).to(device)

In [ ]:
X_train_loader = DataLoader(
    TensorDataset(X_train_tensor, X_train_tensor),
    batch_size=64,
    shuffle=True
)

torch.manual_seed(42)

model_20 = create_denoising_ae(0.20)
optimizer = optim.NAdam(model_20.parameters(), lr=1e-3)
mse = nn.MSELoss()

epochs = 10

for epoch in range(epochs):
    model_20.train()
    total_loss = 0

    for x_batch, y_batch in X_train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model_20(x_batch)
        loss = mse(preds, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: MSE = {total_loss / len(X_train_loader):.5f}")

In [ ]:
torch.manual_seed(42)

model_10 = create_denoising_ae(0.10)
optimizer = optim.NAdam(model_10.parameters(), lr=1e-3)

for epoch in range(epochs):
    model_10.train()
    total_loss = 0

    for x_batch, y_batch in X_train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model_10(x_batch)
        loss = mse(preds, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: MSE = {total_loss / len(X_train_loader):.5f}")

In [ ]:
# Select 5 random images
torch.manual_seed(42)
indices = torch.randperm(len(X_train_tensor))[:5]
samples = X_train_tensor[indices].to(device)

model_20.eval()
model_10.eval()

with torch.no_grad():
    preds_20 = model_20(samples)
    preds_10 = model_10(samples)

fig, axes = plt.subplots(3, 5, figsize=(12, 7))

for i in range(5):
    axes[0, i].imshow(samples[i].cpu().squeeze(), cmap="gray")
    axes[1, i].imshow(preds_20[i].cpu().squeeze(), cmap="gray")
    axes[2, i].imshow(preds_10[i].cpu().squeeze(), cmap="gray")

    for row in range(3):
        axes[row, i].axis("off")

axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Dropout 0.20")
axes[2, 0].set_ylabel("Dropout 0.10")

plt.suptitle("MNIST Denoising Autoencoder Comparison")
plt.tight_layout()
plt.show()

In [ ]:
print("Dropout 0.20 MSE:", mse(preds_20, samples).item())
print("Dropout 0.10 MSE:", mse(preds_10, samples).item())